In [1]:
# ── Cell 1: Clone repo ──────────────────────────────────────────────────────
!git clone -b main https://github.com/ferhat00/crude-oil-forecast.git /kaggle/working/crude-oil-forecast 2>&1 | tail -3

Cloning into '/kaggle/working/crude-oil-forecast'...


In [2]:
# ── Cell 2: Set working directory & Python path ─────────────────────────────
import os, sys

REPO = "/kaggle/working/crude-oil-forecast"
os.chdir(REPO)
sys.path.insert(0, REPO)
print("Working dir:", os.getcwd())

Working dir: /kaggle/working/crude-oil-forecast


In [3]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 110.9 MB/s eta 0:00:0000:010:01


In [4]:
# ── Cell 4: Load API keys from Kaggle Secrets ───────────────────────────────
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["EIA_API_KEY"]          = secrets.get_secret("EIA_API_KEY")
os.environ["FRED_API_KEY"]         = secrets.get_secret("FRED_API_KEY")
os.environ["ALPHA_VANTAGE_API_KEY"] = secrets.get_secret("ALPHA_VANTAGE_API_KEY")
print("API keys loaded.")

API keys loaded.


In [ ]:
# ── Cell 5: Copy config.yaml from private Kaggle dataset ────────────────────
import os, shutil

REPO = "/kaggle/working/crude-oil-forecast"
CONFIG_DATASET_PATH = "/kaggle/input/datasets/ferhat00/crude-oil-config-v4/config.yaml" ##run fast mode
#CONFIG_DATASET_PATH = "/kaggle/input/datasets/ferhat00/crude-oil-config-t/config_t.yaml" ### run thorough mode, 
CONFIG_DEST         = f"{REPO}/config.yaml"

if os.path.exists(CONFIG_DATASET_PATH):
    shutil.copy(CONFIG_DATASET_PATH, CONFIG_DEST)
    print(f"config.yaml copied → {CONFIG_DEST}")
else:
    print("WARNING: config.yaml dataset not attached — falling back to env vars.")
    print("Attach your private dataset via: Notebook → Add Data")

config.yaml copied → /kaggle/working/crude-oil-forecast/config.yaml


In [6]:
# ── Cell 6: Fix yfinance cache issues (Kaggle) ───────────────────────────────
import tempfile
import yfinance as yf

# Redirect timezone cache to avoid file conflict
yf.set_tz_cache_location(tempfile.mkdtemp())

# Disable the SQLite price cache entirely to prevent database lock errors
import yfinance.cache as yf_cache
yf_cache.set_cache_location(tempfile.mkdtemp())

print("yfinance cache redirected.")

yfinance cache redirected.


In [7]:
# ── Cell 7: Run the full pipeline ───────────────────────────────────────────
!python scripts/run_pipeline.py
#!python scripts/run_pipeline.py --skip-download
#!python scripts/run_pipeline.py --skip-eda
#!python scripts/run_pipeline.py --skip-download --skip-eda

2026-05-15 07:33:43 [INFO] pipeline: ============================================================
2026-05-15 07:33:43 [INFO] pipeline: STEP 1: Data Acquisition
2026-05-15 07:33:43 [INFO] pipeline: ============================================================
2026-05-15 07:33:43 [INFO] src.data_acquisition: Fetching oil prices for ['CL=F', 'BZ=F'] from 2015-01-01 to today
2026-05-15 07:33:43 [INFO] yfinance: Failed to create TzCache, reason: Error creating TzCache folder: '/root/.cache/py-yfinance' reason: [Errno 17] File exists: '/root/.cache/py-yfinance'. TzCache will not be used. Tip: You can direct cache to use a different location with 'set_tz_cache_location(mylocation)'
[*********************100%***********************]  2 of 2 completed
2026-05-15 07:33:44 [INFO] src.data_acquisition: Saved oil prices to /kaggle/working/crude-oil-forecast/data/raw/oil_prices.parquet (2860 rows)
2026-05-15 07:33:44 [INFO] src.data_acquisition: Fetching FRED series: ['usd_index', 'cpi', 'fed_funds',

In [8]:
import pandas as pd
df = pd.read_parquet("data/processed/features.parquet")
print(df.shape)  # need > 800 rows for current config


(743, 338)


In [ ]:
# ── Cell 8: Email results after scheduled run ────────────────────────────────
import smtplib, os, json, math, datetime as _dt
from pathlib import Path
import numpy as np
import joblib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from kaggle_secrets import UserSecretsClient

OUT  = Path("/kaggle/working/crude-oil-forecast/outputs")
FIGS = OUT / "figures"


def _fmt(v, spec=".4f", default="N/A"):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return default
        return format(v, spec)
    except (TypeError, ValueError):
        return str(v)


def _load_json(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())
    except Exception as e:
        return {"_error": f"failed to read {p.name}: {e}"}


def _gam_summary_and_top_features(top_n=15):
    """Return (stats_lines, feature_lines) extracted from the saved GAM model."""
    model_path = OUT / "models" / "gam_model.pkl"
    names_path = OUT / "models" / "feature_names.pkl"
    if not model_path.exists():
        return ["GAM model artefact not found."], []
    gam = joblib.load(model_path)
    names = joblib.load(names_path) if names_path.exists() else []
    s = getattr(gam, "statistics_", {}) or {}
    pseudo_r2 = (s.get("pseudo_r2") or {}).get("explained_deviance", float("nan"))
    n_terms = len(names)

    stats_lines = [
        f"  GCV             : {_fmt(s.get('GCV'), '.6f')}",
        f"  AIC             : {_fmt(s.get('AIC'))}",
        f"  AICc            : {_fmt(s.get('AICc'))}",
        f"  Pseudo R^2      : {_fmt(pseudo_r2)}",
        f"  Scale (sigma^2) : {_fmt(s.get('scale'))}",
        f"  N (samples)     : {s.get('n_samples', 'N/A')}",
        f"  N (selected feats): {n_terms}",
    ]

    p_values  = list(s.get("p_values", []) or [])
    edof_term = list(s.get("edof_per_term", []) or [])
    lam_vals  = np.array(getattr(gam, "lam", []) or []).flatten().tolist()

    if not names or not p_values:
        return stats_lines, ["Term-level statistics unavailable."]

    # Align lengths (pyGAM sometimes appends an intercept row)
    m = min(len(names), len(p_values))
    rows = []
    for i in range(m):
        rows.append({
            "name": names[i],
            "p":    float(p_values[i]) if p_values[i] is not None else float("nan"),
            "edof": float(edof_term[i]) if i < len(edof_term) else float("nan"),
            "lam":  float(lam_vals[i])  if i < len(lam_vals)  else float("nan"),
        })

    # Rank by p-value ascending (most significant first); ties broken by EDoF desc
    rows.sort(key=lambda r: (r["p"] if not math.isnan(r["p"]) else 1.0, -r["edof"]))
    top = rows[:top_n]

    header = f"  {'Rank':>4}  {'Feature':<32}  {'p-value':>10}  {'EDoF':>6}  {'lambda':>10}"
    feat_lines = [header, "  " + "-" * (len(header) - 2)]
    for i, r in enumerate(top, 1):
        nm = r["name"] if len(r["name"]) <= 32 else r["name"][:29] + "..."
        feat_lines.append(
            f"  {i:>4}  {nm:<32}  {_fmt(r['p'], '.3e'):>10}  "
            f"{_fmt(r['edof'], '.2f'):>6}  {_fmt(r['lam'], '.2e'):>10}"
        )
    return stats_lines, feat_lines


def _format_backtest(scores):
    if not scores:
        return ["Walk-forward backtest results not found (run without --skip-backtest)."]
    keys_point = [
        ("rmse_price", "RMSE (price)"),
        ("mae_price", "MAE (price)"),
        ("rmse_naive", "RMSE naive"),
        ("mae_naive", "MAE naive"),
        ("skill_rmse", "Skill vs naive (RMSE)"),
        ("skill_mae",  "Skill vs naive (MAE)"),
        ("directional_accuracy", "Directional accuracy"),
    ]
    keys_prob = [k for k in scores if k.endswith("_model")]
    keys_cov  = [k for k in scores if k.startswith("coverage_")]

    lines = []
    for k, label in keys_point:
        if k in scores:
            lines.append(f"  {label:<24}: {_fmt(scores[k])}")
    if keys_prob:
        lines.append("  -- probabilistic (model space)")
        for k in sorted(keys_prob):
            lines.append(f"  {k:<24}: {_fmt(scores[k])}")
    if keys_cov:
        lines.append("  -- empirical PI coverage")
        for k in sorted(keys_cov):
            lines.append(f"  {k:<24}: {_fmt(scores[k])}")
    return lines


def build_email_body():
    forecast  = _load_json(OUT / "forecast.json") or {}
    bt_scores = _load_json(OUT / "backtest_path.scores.json") or {}
    combo     = _load_json(OUT / "combination_weights.json") or {}
    gam_stats, top_feats = _gam_summary_and_top_features(top_n=15)

    now_utc = _dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")

    L = []
    L.append("Crude Oil Forecast — Daily Pipeline Report")
    L.append(f"Generated: {now_utc}")
    L.append("=" * 60)
    L.append("")
    L.append("== Next-day Forecast ==")
    if forecast:
        L.append(f"  Last trading day : {forecast.get('last_trading_day','N/A')}")
        L.append(f"  Last close       : ${_fmt(forecast.get('last_close'), '.2f')}")
        L.append(f"  Forecast date    : {forecast.get('forecast_date','N/A')}")
        L.append(f"  Target space     : {forecast.get('target_transform','N/A')}")
        L.append(f"  Point forecast   : ${_fmt(forecast.get('point_forecast'), '.2f')}")
        L.append(f"  Sigma (price)    : ${_fmt(forecast.get('sigma'), '.2f')}")
        L.append(f"  95% PI           : ${_fmt(forecast.get('pi_95_lower'), '.2f')} — "
                 f"${_fmt(forecast.get('pi_95_upper'), '.2f')}")
        if "forecast_log_return" in forecast:
            L.append(f"  Forecast log-ret : {_fmt(forecast['forecast_log_return'], '.5f')}")
        if "combo_forecast" in forecast:
            L.append(f"  Combo forecast   : ${_fmt(forecast.get('combo_forecast'), '.2f')}  "
                     f"(w_gam = {_fmt(forecast.get('combo_weight_gam'), '.4f')})")
    else:
        L.append("  forecast.json not found.")
    L.append("")

    L.append("== GAM Model Statistics ==")
    L.extend(gam_stats)
    L.append("")

    L.append("== Walk-Forward Backtest ==")
    L.extend(_format_backtest(bt_scores))
    L.append("")

    if combo:
        L.append("== Bates-Granger Combination Weights ==")
        for k, v in combo.items():
            L.append(f"  {k:<24}: {_fmt(v) if isinstance(v, (int, float)) else v}")
        L.append("")

    L.append("== Top Features (ranked by p-value) ==")
    L.extend(top_feats)
    L.append("")

    L.append("== Attached Artefacts ==")
    L.append("  - fan_chart.png           : 95% predictive bands over recent history")
    L.append("  - predictive_density.png  : next-day predictive distribution")
    L.append("  - actual_vs_pred.png      : in-sample fit diagnostic")
    L.append("")
    L.append("Outputs on Kaggle: /kaggle/working/crude-oil-forecast/outputs/")
    return "\n".join(L)


def send_email_with_attachment(sender_email, sender_password, receiver_email,
                                subject, body, file_paths):
    msg = MIMEMultipart()
    msg["From"]    = sender_email
    msg["To"]      = receiver_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    for file_path in file_paths:
        if not os.path.exists(file_path):
            print(f"WARN: attachment missing, skipping: {file_path}")
            continue
        with open(file_path, "rb") as f:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition",
                        f"attachment; filename={os.path.basename(file_path)}")
        msg.attach(part)

    # Port 465 + SSL — works on Kaggle scheduled runs
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(sender_email, sender_password)
        server.send_message(msg)
    print(f"Email sent to {receiver_email}")


secrets         = UserSecretsClient()
sender_email    = secrets.get_secret("GMAIL_ADDRESS")
sender_password = secrets.get_secret("GMAIL_APP_PASSWORD")

body = build_email_body()
print(body)  # echo to notebook output for quick inspection

send_email_with_attachment(
    sender_email    = sender_email,
    sender_password = sender_password,
    receiver_email  = sender_email,
    subject         = "Crude Oil Forecast — Daily Update",
    body            = body,
    file_paths      = [
        str(FIGS / "fan_chart.png"),
        str(FIGS / "predictive_density.png"),
        str(FIGS / "actual_vs_pred.png"),
    ],
)
